In [9]:
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()

True

In [10]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [11]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


In [12]:
import json

def extract_resume_info(client, resume_text):
    prompt = f"""
You are an AI resume evaluator.

Extract the following details from the resume text:
- total_years_experience (number)
- skills (list of strings)
- seniority_level (Junior / Mid / Senior / Lead)
- leadership_role (true or false)
- leadership_evidence (short text)

Return ONLY valid JSON. No explanation.

Resume Text:
{resume_text}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a resume analysis agent."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content
    return json.loads(content)


In [13]:
def calculate_score(data):
    score = 0

    # Experience
    exp = data.get("total_years_experience", 0)
    if exp >= 10:
        score += 30
    elif exp >= 5:
        score += 20
    else:
        score += 10

    # Skills
    skills = data.get("skills", [])
    score += min(len(skills) * 3, 30)

    # Seniority
    seniority_score = {
        "Junior": 5,
        "Mid": 10,
        "Senior": 20,
        "Lead": 30
    }
    score += seniority_score.get(data.get("seniority_level"), 0)

    # Leadership
    if data.get("leadership_role"):
        score += 10

    return min(score, 100)


In [17]:
#from pdf_reader import extract_text_from_pdf


#load_dotenv()
#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

pdf_path = "DE_Maheswar_Nayak_VISA_Data_Engg.pdf"

resume_text = extract_text_from_pdf(pdf_path)
resume_data = extract_resume_info(client, resume_text)

final_score = calculate_score(resume_data)
resume_data["final_score"] = final_score

print(resume_data)


{'total_years_experience': 10, 'skills': ['Data Pipeline Development (Spark, Airflow, Hive)', 'Cloud Data Platforms (Google Cloud Platform, Big Query, Cloud Composer, Dataproc)', 'Data Modeling & Transformation', 'SQL', 'PLSQL', 'Python', 'PySpark', 'Data Integration & ETL (Extract, Transform, Load)', 'Performance Optimization & Troubleshooting', 'Business Requirements Analysis & Collaboration', 'Data Quality & Validation'], 'seniority_level': 'Senior', 'leadership_role': False, 'leadership_evidence': '', 'final_score': 80}
